In [198]:
import pandas as pd 
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score , recall_score , precision_score , f1_score

In [199]:
df = pd.read_csv('student_placement_career_success_dataset.csv')

In [200]:
df.info()
df.describe()
df.columns

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 44 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   student_id                  25000 non-null  str    
 1   age                         25000 non-null  int64  
 2   gender                      25000 non-null  str    
 3   state                       25000 non-null  str    
 4   city_tier                   25000 non-null  int64  
 5   college_tier                25000 non-null  int64  
 6   branch                      25000 non-null  str    
 7   degree                      25000 non-null  str    
 8   cgpa                        25000 non-null  float64
 9   backlog_history             25000 non-null  int64  
 10  attendance_percentage       25000 non-null  float64
 11  primary_language            25000 non-null  str    
 12  DSA_problems_solved         25000 non-null  int64  
 13  GitHub_repos                24500 non-null

Index(['student_id', 'age', 'gender', 'state', 'city_tier', 'college_tier',
       'branch', 'degree', 'cgpa', 'backlog_history', 'attendance_percentage',
       'primary_language', 'DSA_problems_solved', 'GitHub_repos',
       'hackathons_participated', 'development_projects_count',
       'AI_ML_projects', 'internships_completed', 'resume_score',
       'communication_skills', 'aptitude_score', 'mock_interview_score',
       'sleep_hours', 'screen_time', 'gaming_hours', 'study_hours_daily',
       'stress_level', 'burnout_score', 'gym_frequency', 'family_income_lpa',
       'self_learning_hours', 'motivation_level', 'placement_status',
       'company_type', 'work_mode', 'salary_lpa', 'interview_rounds_cleared',
       'offer_count', 'joining_delay_months', 'layoffs_risk_score',
       'AI_tool_usage_frequency', 'prompt_engineering_skill', 'AI_fear_score',
       'adaptability_score'],
      dtype='str')

In [201]:
df.isnull().sum()
df.duplicated().sum()

np.int64(0)

----->>>>> MANAGE NULL VALUE <<<<<------

In [202]:
col_null = df.columns[df.isnull().any()]
print(col_null)

Index(['GitHub_repos', 'mock_interview_score', 'sleep_hours', 'company_type',
       'work_mode'],
      dtype='str')


In [203]:
df[col_null].head()

,GitHub_repos,mock_interview_score,sleep_hours,company_type,work_mode
0,10.0,NaN,5.7,Startup,Remote
1,16.0,80.0,5.3,MNC,Onsite
2,30.0,81.0,4.9,Startup,Remote
3,15.0,100.0,6.4,Product-Based,Remote
4,12.0,68.0,4.8,Startup,Onsite


In [204]:
col_null_cat = df[col_null].select_dtypes(include='str').columns
col_null_num = df[col_null].select_dtypes(include = 'number').columns
print(col_null_cat)
print(col_null_num)

Index(['company_type', 'work_mode'], dtype='str')
Index(['GitHub_repos', 'mock_interview_score', 'sleep_hours'], dtype='str')


In [205]:
for col in col_null_num : 
    df[col] = df[col].fillna(df[col].mean())

for col in col_null_cat :
    df[col] = df[col].fillna(df[col].mode()[0])

In [206]:
df = df.drop("student_id", axis=1)

----->>>>> ENCODING <<<<<-----

In [207]:
col_encoding = df.select_dtypes(include='str').columns
print(col_encoding)

Index(['gender', 'state', 'branch', 'degree', 'primary_language',
       'placement_status', 'company_type', 'work_mode'],
      dtype='str')


In [208]:
col_le = ['gender' , 'placement_status' ]
col_ohe = ['state' , 'branch' , 'degree' , 'primary_language' ,'company_type' , 'work_mode']

In [209]:
le = LabelEncoder()
for col in col_le :
    df[col] = le.fit_transform(df[col]) 

df = pd.get_dummies(df , columns=col_ohe, drop_first=True , dtype=int)

In [210]:
df.shape

(25000, 61)

In [211]:
corr = df.corr()["placement_status"].sort_values(ascending=False)
print(corr.sample(34))


aptitude_score                0.025252
state_Karnataka               0.005145
adaptability_score           -0.005859
interview_rounds_cleared      0.218522
DSA_problems_solved           0.109216
state_Maharashtra             0.004831
degree_BTech                 -0.001819
gender                        0.000724
resume_score                  0.120278
mock_interview_score          0.016422
company_type_Product-Based    0.073386
development_projects_count    0.103924
motivation_level              0.002824
stress_level                  0.005084
state_Uttar Pradesh           0.007383
company_type_Service-Based    0.072439
burnout_score                -0.037434
college_tier                 -0.047143
branch_Mechanical             0.008111
AI_fear_score                -0.001298
state_Gujarat                 0.007749
internships_completed         0.124653
branch_Electronics           -0.014205
layoffs_risk_score           -0.150657
age                           0.006884
communication_skills     

In [212]:
df = df.drop(corr[abs(corr) < 0.01].index, axis=1)

In [213]:
df.shape

(25000, 31)

----->>>>> TRAIN TEST SPLIT <<<<<------

In [214]:
leakage_cols = [
    "offer_count",
    "salary_lpa",
    "interview_rounds_cleared",'company_type_Product-Based',
    'company_type_Service-Based',
    'company_type_Startup',
    'work_mode_Onsite',
    'work_mode_Remote'
]
df = df.drop(columns=leakage_cols)

x = df.drop(columns=["placement_status"])
y = df["placement_status"]

In [215]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.33, random_state=42)


---->>>>>> STANDARD SCALER <<<<<-----

In [216]:
scaler = StandardScaler()
x_train_scaler  = scaler.fit_transform(x_train)
x_test_scaler = scaler.transform(x_test)

------>>>>> LOGISTIC REGRESSION <<<<<------

In [217]:
model = LogisticRegression(max_iter=1500)
model.fit(x_train_scaler , y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [218]:
y_lr = model.predict(x_test_scaler)

In [219]:
print('Accuracy Score of Logistic Regression: ' , accuracy_score(y_test , y_lr))
print('Recall Score of Logistic Regression: ' , recall_score(y_test , y_lr))
print('Precision Score of Logistic Regression: ' , precision_score(y_test , y_lr))
print('F1 Score of Logistic Regression: ' , f1_score(y_test , y_lr))

Accuracy Score of Logistic Regression:  0.9878787878787879
Recall Score of Logistic Regression:  0.9963022309873043
Precision Score of Logistic Regression:  0.9914142033607262
F1 Score of Logistic Regression:  0.9938522070576663


----->>> KNN <<<----

In [220]:
model_knn = KNeighborsClassifier()
model_knn.fit(x_train_scaler , y_train)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.Doesn't affect :meth:`fit` method.",None


In [221]:
y_knn = model_knn.predict(x_test_scaler)

In [222]:
print('Accuracy Score of knn: ' , accuracy_score(y_test , y_knn))
print('Recall Score of KNN: ' , recall_score(y_test , y_knn))
print('Precision Score of KNN: ' , precision_score(y_test , y_knn))
print('F1 Score of KNN: ' , f1_score(y_test , y_knn))

Accuracy Score of knn:  0.9843636363636363
Recall Score of KNN:  0.9996302230987304
Precision Score of KNN:  0.9847013113161729
F1 Score of KNN:  0.9921096091504068


------>>>>> Naive Bayes <<<<<-------

In [223]:
model_nb = GaussianNB()
model_nb.fit(x_train_scaler , y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [224]:
y_nb = model_nb.predict(x_test_scaler)

In [225]:
print('Accuracy Score of Naive Bayes: ' , accuracy_score(y_test , y_nb))
print('Recall Score of Naive Bayes: ' , recall_score(y_test , y_nb))
print('Precision Score of Naive Bayes: ' , precision_score(y_test , y_nb))
print('F1 Score of Naive Bayes: ' , f1_score(y_test , y_nb))

Accuracy Score of Naive Bayes:  0.8968484848484849
Recall Score of Naive Bayes:  0.8953531369407124
Precision Score of Naive Bayes:  0.9997247453894853
F1 Score of Naive Bayes:  0.9446648026529684


----->>>>>> Decision Tree <<<<<-----

In [226]:
model_dt = DecisionTreeClassifier()
model_dt.fit(x_train_scaler , y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the curre

In [227]:
y_dt = model_dt.predict(x_test_scaler)

In [228]:
print('Accuracy Score of D: ' , accuracy_score(y_test , y_nb))
print('Recall Score of Naive Bayes: ' , recall_score(y_test , y_nb))
print('Precision Score of Naive Bayes: ' , precision_score(y_test , y_nb))
print('F1 Score of Naive Bayes: ' , f1_score(y_test , y_nb))

Accuracy Score of D:  0.8968484848484849
Recall Score of Naive Bayes:  0.8953531369407124
Precision Score of Naive Bayes:  0.9997247453894853
F1 Score of Naive Bayes:  0.9446648026529684


In [233]:
import pickle

# Model aur Scaler ko save kar rahe hain
with open('placement_model.pkl', 'wb') as model_file:
    pickle.dump(models["Logistic Regression"], model_file)  # Ya fir models["KNN"] jo tumhe best lage

with open('scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

# Un columns ki list bhi save kar lete hain jo model ko input chahiye
with open('features.pkl', 'wb') as features_file:
    pickle.dump(list(x_train.columns), features_file)

print("Model, Scaler aur Features successfully save ho gaye hain!")

Model, Scaler aur Features successfully save ho gaye hain!
